<a href="https://colab.research.google.com/github/leon72305-source/26007815_tb1/blob/main/26007815_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UFCE3R-30-3 - Big Data Analytics
# TB1 Final Assessment (25 marks)

For this assessment, you will need Python and PySpark. We strongly recommend using Google Colab. You may use UWE PCs as well, as we have already covered these topics in both environments. However, we cannot guarantee that automatic updates on UWE PCs will keep library versions aligned.

The assessment has four sections:

- **Section A** - Descriptive analytics (Python/pandas)
- **Section B** - Data visualisation (matplotlib & seaborn)
- **Section C** - Diagnostic analytics: anomaly detection and correlation analysis (Python)
- **Section D** - Big data analysis with PySpark

### Notice 1
Please DO NOT change (add/remove) cells in this notebook. Only write your codes and your comments/reflections in the specified cells. Changing the layout of the notebook/cells, will affect your mark. As stated in the assessment brief, the total number of words in Markdown cells in this Jupyter Notebook needs to be within the specified limit. Please note that this notebook already holds 3333 words in its Markdown cells. When counting words, ignore this amount. You are provided a Jupyter Notebook named `Markdown_word_count.ipynb` to count the number of words in Markdown cells.

### Notice 2
Where requested, add at least 1 test case to verify correctness of your code. Add test cases ONLY in the provided cells.

### Notice 3
To achieve higher marks (optional), you would create a repository (such as in Github) for this assessment and commit gradual progress there. Share the link of the repo in the next cell. If your repo is private, share it with the email of all lecturers:

`Xiaodong.Li@uwe.ac.uk`

`Alireza.Bolhari@uwe.ac.uk`

### Notice 4
Section D uses a dataset with several million rows. Do NOT load it with pandas: this may crash your session and it will not be marked as a big data solution.

==================================================

## Please write your full name and student ID here:

### Full name: Alexander Leonard Gotama
### Student ID: 26007815
### Repo URL: https://github.com/leon72305-source/26007815_tb1

==================================================

## *Datasets Description*

For this assessment, you are going to work with these 2 datasets:
1) CSV: hourly electricity generation, load and prices in Spain (`spain_energy_hourly.csv`), used in Sections A, B and C
2) CSV / PARQUET (two identical formats for this dataset. You can choose which one to work on): fuel prices at petrol stations across France (`fuel_in_france_2026_01_to_2026_09.csv` or `fuel_in_france_2026_01_to_2026_09.parquet`), used in Section D

You can download these datasets from BlackBoard.

A brief description about these datasets:

- 1: this dataset holds 4 years of hourly data (2015-2018) about the Spanish electricity system. Each row is one hour. Columns starting with `generation` hold the electricity generated by each source (MW). `forecast ... day ahead` columns hold the forecasts made one day before. `total load forecast` and `total load actual` hold the forecast and actual electricity demand (MW). `price day ahead` and `price actual` hold electricity prices (EUR/MWh).

Link to the original dataset: https://huggingface.co/datasets/vitaliy-sharandin/energy-consumption-hourly-spain

- 2: this dataset holds fuel price records from petrol stations across France, from January to September 2026 (several million rows). Each row is one price record of one fuel type at one station, together with the Brent crude oil price. The original dataset is updated daily. Column names are in French, so use the data dictionary below.

Link to the original dataset: https://huggingface.co/datasets/VincentGOURBIN/FuelInFranceData

| Column | Meaning |
| ----------- | ----------- |
| `rate_date` | Date and time of the price record (`dd/MM/yyyy HH:mm`) |
| `station_id` | Unique identifier of the station |
| `nom` | Name of the station |
| `commune` | Town / municipality |
| `marque` | Brand of the station |
| `departement` | French department code (e.g. `01`, `2A`) |
| `regioncode` | French region code |
| `zipcode` | Postcode (French postcodes have 5 characters and may start with `0`) |
| `address` | Street address |
| `coordlatitude`, `coordlongitude` | Location of the station |
| `fuel_name` | Fuel type (`Gazole` = diesel, `SP95`, `SP98`, `E10`, `E85`, `GPLc` = LPG) |
| `price` | Price of the fuel (EUR per litre) |
| `brent_date` | Date of the Brent crude oil price |
| `brent_rate` | Brent crude oil price (USD per barrel) |
| `brent_rate_eur` | Brent crude oil price (EUR per barrel) |
| `source_file` | Name of the monthly file the record was collected from |

## Uploading datasets to Colab

You may upload `.zip` version of the datasets to Colab and then unzip them using Python code. For instance, to unzip `file_name.zip`:

```
import zipfile

with zipfile.ZipFile('file_name.zip', 'r') as zip_ref:
    zip_ref.extractall('.')
```

As `fuel_in_france_2026_01_to_2026_09.zip` is a large file, give enough time for the upload to finish. Alternatively, you may put it in your Google Drive and mount the drive in Colab.

In [ ]:
# If you need to unzip files
from google.colab import drive
import zipfile

drive.mount("/content/drive/")
folder_path = "/content/drive/MyDrive/DataSet/"

zip_files = [
    "fuel_in_france_2026_01_to_2026_09_csv.zip",
    "fuel_in_france_2026_01_to_2026_09_parquet.zip",
    "spain_energy_hourly.zip"
]

for zip_name in zip_files:
  zip_path = folder_path + zip_name
  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(folder_path)
    print(f"Successfully extracted: {zip_name}")

## Environment Setup

Please ensure you have PySpark installed. If you need to install it (or any other library), use this cell. The `-q` flag keeps the output quiet.

In [ ]:
# BEGIN SOLUTION
# Installs PySpark
!pip install -q pyspark

# END SOLUTION

If you need to `import` anything, please ONLY use this cell to put ALL the `import` statements for this notebook:

In [ ]:
# Put ALL the import statements for this notebook in this cell.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession


# Section A: Descriptive Analytics

Descriptive analytics is the first step in understanding data: summarising and presenting it in a meaningful way. In this section you will use pandas and the `spain_energy_hourly.csv` dataset.

## 1: Loading the energy dataset

Write a Python code to load `spain_energy_hourly.csv` into a pandas DataFrame named `df_energy`. Convert the `time` column into a proper datetime type and set it as the index of the DataFrame. Then display the first 10 and the last 10 rows, and print the shape (number of rows and columns) of the DataFrame.

Note: the `time` column holds timestamps with different UTC offsets (`+01:00` in winter and `+02:00` in summer). Something like `pd.to_datetime(..., utc=True)` followed by `.tz_convert('Europe/Madrid')` may help.

In [ ]:
# BEGIN SOLUTION
energy_path = folder_path + "spain_energy_hourly.csv"
df_energy = pd.read_csv(energy_path) # Loads'spain_energy_hourly.csv'as df_energy

# Converts the different timezones into a single timeline
df_energy['time'] = pd.to_datetime(df_energy['time'], utc=True).dt.tz_convert('Europe/Madrid')
df_energy.set_index('time', inplace=True) # Converts integer index into a row index

display("First 10 rows:", df_energy.head(10))
display("Last 10 rows:", df_energy.tail(10))
print("Dataframe shape (rows, columns):",df_energy.shape) # Displays total number of rows and columns

# END SOLUTION

In [ ]:
# Checks if variable df_energy is a pd dataframe
assert isinstance(df_energy, pd.DataFrame), "Error: df_energy is not a pandas dataframe"

#Extracts timezone info and turns it into string like EU/madrid to compare
assert str(df_energy.index.tz) == 'Europe/Madrid', "Error: df_energy.index is not in EU/Madrid timezone"
print("All test cases passed, dataset is properly loaded!")

## 2: Data types and data quality audit

Using `df_energy`:
- print the data type of each column
- print the number AND the percentage of missing values in each column
- identify the columns that are **entirely** missing and the columns whose values are **all zero**
- check whether there are any duplicate timestamps

Store the names of the columns that hold no useful information in a list named `cols_to_drop`, then drop them and save the result in a new DataFrame named `df_clean`.

In [ ]:
# BEGIN SOLUTION
print("Column Data Types\n", df_energy.dtypes) #Displays data types for every column
print("\nTotal Missing Values\n", df_energy.isnull().sum()) #Displays total missing values in each column
print("\nPercentage of Missing Values\n", (df_energy.isnull().mean() * 100).round(2)) #Displays percentage of missing values

entirely_missing = [col for col in df_energy.columns if df_energy[col].isnull().all()]
all_zero = [col for col in df_energy.columns if (df_energy[col] == 0).all()]

print("\nColumns entirely missing:", entirely_missing)
print("Columns with all zero values:", all_zero)
print("Duplicate timestamps count:", df_energy.index.duplicated().sum())

cols_to_drop = list(set(entirely_missing + all_zero))
df_clean = df_energy.drop(columns=cols_to_drop)

print("Dropped columns:", cols_to_drop)
print("Remaining columns in df_clean:", list(df_clean.columns))
# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.
for col in cols_to_drop:
    assert df_energy[col].isnull().all() or (df_energy[col] == 0).all(), f"Column '{col}' is neither all null or all zero!"

for col in cols_to_drop:
    assert col not in df_clean.columns, f"Dropped '{col}' is still present in df_clean!"
print("All test cases passed, dataset is properly loaded!")

What data quality issues did you find? Why is it justified to drop the columns in `cols_to_drop`? Write your answer in the Markdown cell below.

#### START.
Your analysis / interpretation goes here:

There are plenty of columns with slightly missing values which might impact the reliability of test results. However, `cols_to_drop` is empty so removing it won't impact the validity of data results produced, as no other columns are relying on values from the columns dropped.


#### END.

## 3: Handling the remaining missing values

Some columns in `df_clean` might still have a small number of missing values. Handle them using a method that is appropriate for **hourly time-series** data. Print the number of missing values in each column before and after your treatment.

In [19]:
# BEGIN SOLUTION
print("Values Before Treatment\n", df_clean.isnull().sum()) #Displays total missing values in each column
df_clean = df_clean.interpolate(method='linear').ffill().bfill() # Handle missing values using linear interpolation and fill remaining edges
print("\nMissing Values After Treatment\n", df_clean.isnull().sum())
# END SOLUTION

Values Before Treatment
 generation biomass                             0
generation fossil brown coal/lignite           0
generation fossil coal-derived gas             0
generation fossil gas                          0
generation fossil hard coal                    0
generation fossil oil                          0
generation fossil oil shale                    0
generation fossil peat                         0
generation geothermal                          0
generation hydro pumped storage consumption    0
generation hydro run-of-river and poundage     0
generation hydro water reservoir               0
generation marine                              0
generation nuclear                             0
generation other                               0
generation other renewable                     0
generation solar                               0
generation waste                               0
generation wind offshore                       0
generation wind onshore                     

In [17]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.

#Checks for missing and zero values in df_clean
assert df_clean.isnull().all().sum() == 0, "Error: df_clean has missing values"
print("All test cases passed, dataset is properly loaded!")

All test cases passed, dataset is properly loaded!


Why did you choose this method? Why could filling with the column mean, or dropping the rows, be less appropriate here? Write your answer in the Markdown cell below.

#### START.
Your analysis / interpretation goes here:

Dropping values could impact the validity of the table if another value relies on it. Additionally, replacing flat values may impact the natural variance that distorts the dataset.

#### END.

## 4: Summary statistics

For `df_clean`:
- calculate summary statistics (count, mean, standard deviation, minimum, quartiles, maximum) for all numerical columns
- calculate the mode and the range of `total load actual` and `price actual`
- calculate the skewness of `price actual`

In [28]:
# BEGIN SOLUTION
# Your code here with comments or explanation
summary_stats = df_clean.describe()
display(summary_stats)

target_cols = ['total load actual', 'price actual']
for col in target_cols:
  mode_val = df_clean[col].mode().values
  range_val = df_clean[col].max() - df_clean[col].min()
  print(f"Mode of '{col}': {mode_val}")
  print(f"Range of '{col}': {range_val}\n")

price_skewness = df_clean['price actual'].skew()
print(f"Skewness of 'price actual': {price_skewness}")

# END SOLUTION

,generation biomass,generation fossil brown coal/lignite,generation fossil coal-derived gas,generation fossil gas,generation fossil hard coal,generation fossil oil,generation fossil oil shale,generation fossil peat,generation geothermal,generation hydro pumped storage consumption,...,generation solar,generation waste,generation wind offshore,generation wind onshore,forecast solar day ahead,forecast wind onshore day ahead,total load forecast,total load actual,price day ahead,price actual
count,35064.000000,35064.000000,35064.0,35064.000000,35064.000000,35064.000000,35064.0,35064.0,35064.0,35064.000000,...,35064.000000,35064.000000,35064.0,35064.000000,35064.000000,35064.000000,35064.000000,35064.000000,35064.000000,35064.000000
mean,383.531343,448.094570,0.0,5622.700647,4256.531271,298.342417,0.0,0.0,0.0,475.582706,...,1432.818546,269.418692,0.0,5464.980450,1439.066735,5471.216689,28712.129962,28698.281385,49.874341,57.884023
std,85.346266,354.618269,0.0,2201.510984,1961.988937,52.519927,0.0,0.0,0.0,792.312704,...,1679.961733,50.218423,0.0,3213.586296,1677.703355,3176.312853,4594.100854,4575.828854,14.618900,14.204083
min,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.000000,...,0.000000,0.000000,0.0,0.000000,0.000000,237.000000,18105.000000,18041.000000,2.060000,9.330000
25%,333.000000,0.000000,0.0,4126.000000,2527.000000,263.000000,0.0,0.0,0.0,0.000000,...,71.000000,240.000000,0.0,2933.000000,69.000000,2979.000000,24793.750000,24807.000000,41.490000,49.347500
50%,367.000000,509.000000,0.0,4969.500000,4475.000000,300.000000,0.0,0.0,0.0,68.000000,...,616.000000,279.000000,0.0,4849.500000,576.000000,4855.000000,28906.000000,28902.000000,50.520000,58.020000
75%,433.000000,757.000000,0.0,6429.000000,5839.000000,330.000000,0.0,0.0,0.0,616.000000,...,2579.000000,310.000000,0.0,7399.500000,2636.000000,7353.000000,32263.250000,32194.250000,60.530000,68.010000
max,592.000000,999.000000,0.0,20034.000000,8359.000000,449.000000,0.0,0.0,0.0,4523.000000,...,5792.000000,357.000000,0.0,17436.000000,5836.000000,17430.000000,41390.000000,41015.000000,101.990000,116.800000


Mode of 'total load actual': [23665.]
Range of 'total load actual': 22974.0

Mode of 'price actual': [56.85]
Range of 'price actual': 107.47

Skewness of 'price actual': -0.32374928779368634


#### START.
Describe the outputs (remember the units: MW and EUR/MWh):

The mode of of total load actual show a frequent electricity demand value across hourly timestamps are 23,665 MW, with its range being 22,974 that showcases a wide range of grid consumption through peak and off-peak times.

The mode of price actual highlights that the most common electricity market price is at €56.85/MWh, while its range is €107.47/MWh that shows moderate market changes

#### END.

## 5: Feature engineering

Add these new columns to `df_clean`:
- `year`, `month`, `hour` and `day_of_week`, derived from the (local time) index
- `fossil_generation` = sum of `generation fossil brown coal/lignite`, `generation fossil gas`, `generation fossil hard coal` and `generation fossil oil`
- `renewable_generation` = sum of `generation biomass`, `generation hydro run-of-river and poundage`, `generation hydro water reservoir`, `generation other renewable`, `generation solar` and `generation wind onshore`
- `total_generation` = sum of all remaining `generation ...` columns, **excluding** `generation hydro pumped storage consumption` (this column is electricity consumed for pumping, not generated)
- `renewable_share` = `renewable_generation` / `total_generation` x 100
- `load_forecast_error` = `total load actual` - `total load forecast`

In [29]:
# BEGIN SOLUTION
#Extracts datettime attributes from local time indexes
df_clean['year'] = df_clean.index.year
df_clean['month'] = df_clean.index.month
df_clean['hour'] = df_clean.index.hour
df_clean['day_of_week'] = df_clean.index.dayofweek

#Fossil Generation Column
fossils_cols = ['generation fossil brown coal/lignite',
                'generation fossil gas',
                'generation fossil hard coal',
                'generation fossil oil']
df_clean['fossil_generation'] = df_clean[fossils_cols].sum(axis=1) #Sums row-by-row instead of using column totals

#Renewable Generation Column
renewable_cols = ['generation biomass',
                  'generation hydro run-of-river and poundage',
                  'generation hydro water reservoir',
                  'generation other renewable',
                  'generation solar',
                  'generation wind onshore']
df_clean['renewable_generation'] = df_clean[renewable_cols].sum(axis=1)

#Total Generation Column
gen_columns = [
    col for col in df_clean.columns
    if col.startswith('generation') and col not in [
        'generation hydro pumped storage consumption',
        'fossil_generation',
        'renewable_generation']
]
df_clean['total_generation'] = df_clean[gen_columns].sum(axis=1)

#Renewable Share Column
df_clean['renewable_share'] = (df_clean['renewable_generation'] / df_clean['total_generation']) * 100
df_clean['renewable_share'] = df_clean['renewable_share'].fillna(0)

#Load Forecast Error Column
df_clean['load_forecast_error'] = df_clean['total load actual'] - df_clean['total load forecast']
print("New columns successfully added!")
# END SOLUTION

New columns successfully added!


In [31]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.
expected_cols = [
    'year', 'month', 'hour', 'day_of_week', 'fossil_generation',
    'renewable_generation', 'renewable_share', 'load_forecast_error'
]
for col in expected_cols:
  assert col in df_clean.columns, f"Error: The column '{col}' is missing from df_clean"

#Ensures percentages are only between 0 to 100%
assert df_clean['renewable_share'].between(0, 100).all(), "Error: renewable_share has invalid values"
print("All test cases passed, there are no more missing values!")

All test cases passed, there are no more missing values!


## 6: Data grouping and aggregation

a) Group the data by `year` and calculate: the mean `price actual`, the mean `total load actual`, the total `renewable_generation` in GWh (hint: each row is one hour, so summing MW gives MWh), and the mean `renewable_share`.

b) Group the data by `hour` and calculate the mean `total load actual` and the mean `price actual`. Which hour of the day has the highest average demand, and which has the highest average price?

c) Which month (year and month) had the highest average `price actual`? Which month had the highest average `renewable_share`?

In [53]:
# BEGIN SOLUTION
#Yearly Grouping
yearly_cols = df_clean.groupby(df_clean.index.year).agg({
    'price actual': 'mean',
    'total load actual': 'mean',
    'renewable_generation': lambda x: x.sum() / 1000, #Sums MWh to convert to GWh
    'renewable_share': 'mean'
}).rename(columns={'renewable_generation': 'renewable_generation_GWh'})
display(yearly_cols.head(5))

#Hourly Grouping
hourly_cols = df_clean.groupby(df_clean.index.hour).agg({
    'total load actual': 'mean',
    'price actual': 'mean'
})
hourly_high_demand = hourly_cols['total load actual'].idxmax()
hourly_high_price = hourly_cols['price actual'].idxmax()
print(f"\nHour with highest average demand: {hourly_high_demand}:00")
print(f"Hour with highest average price: {hourly_high_price}:00 \n")

#High Averages
monthly_cols = df_clean.groupby([df_clean.index.year, df_clean.index.month]).agg({
    'price actual': 'mean',
    'renewable_share': 'mean'
})
month_high_price = tuple(map(int,monthly_cols['price actual'].idxmax()))
month_high_share = tuple(map(int,monthly_cols['renewable_share'].idxmax()))
print(f"Month with highest average price actual: {month_high_price}")
print(f"Month with highest renewable share: {month_high_share}")
# END SOLUTION

,price actual,total load actual,renewable_generation_GWh,renewable_share
time,,,,
2015,61.360812,28362.775571,95524.403,37.505227
2016,47.437169,28507.624658,102532.643,41.458070
2017,59.327340,28858.547945,84818.124,34.365976
2018,63.439393,29064.699715,100889.815,40.573755



Hour with highest average demand: 13:00
Hour with highest average price: 21:00 

Month with highest average price actual: (2017, 1)
Month with highest renewable share: (2018, 3)


#### START.
Describe the outputs:

Hours with highest average power demand can be seen at 13:00, which may be driven by daytime commerical and industrial work, while the highest average electricity price is seen at 21:00, which can be driven for when solar and renewable generation production is at its lowest.


#### END.

# Section B: Data Visualisation

All charts must have a proper title, axis names (with units where applicable), legends where needed and readable tick labels. Use comments (begin with #) to explain your coding work. Use **matplotlib** for questions 7 and 8, and **seaborn** for question 9.

## 7: Time-series chart (matplotlib)

Resample the hourly data to **daily** averages. In one figure with two vertically stacked subplots that share the x-axis, plot:
- top: the daily average `price actual`, together with its 30-day rolling mean
- bottom: the daily average `total load actual`

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

#### START.
Describe the chart:



#### END.

## 8: Generation mix (matplotlib)

In one figure with two subplots, create:
- a **stacked area chart** of the monthly total generation (in GWh) from `generation nuclear`, `fossil_generation`, `renewable_generation` and "other" (everything else in `total_generation`)
- a **pie chart** showing the share of each of these four groups in the total generation over the whole period

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

#### START.
Describe the charts:



#### END.

## 9: Patterns and distributions (seaborn)

Create one figure with a 2 x 2 grid of subplots:
- (a) a box plot of `price actual` for each `month`
- (b) a heatmap of the mean `total load actual` for each `hour` (rows) and `day_of_week` (columns)
- (c) a histogram with a KDE curve of `price actual`
- (d) a scatter plot of `total load actual` (x) against `price actual` (y), coloured by `renewable_share`. Use a random sample of 5,000 rows (`random_state=42`) to keep the chart readable.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

#### START.
Describe the charts:



#### END.

# Section C: Diagnostic Analytics

Diagnostic analytics aims to uncover the "why" behind observed trends, patterns and anomalies. You will continue using `df_clean`.

## 10: Anomaly detection - z-score

Electricity prices change with seasons, so a price that is normal in one month may be unusual in another. For each **year-month**, calculate the z-score of `price actual` relative to the mean and standard deviation of that year-month, and store it in a new column `price_z`. Define "unusual" as hours where `|price_z| > 3`.

- print the total number of unusual hours and the number of unusual hours per year
- show the 10 most extreme unusual hours (largest `|price_z|`)
- for the year with the most unusual hours, plot the hourly `price actual` and highlight the unusual hours in red

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.



## 11: Anomaly detection - IQR and forecast problems

a) Detect unusual `price actual` values again, this time with the IQR method (outside `Q1 - 1.5 x IQR` and `Q3 + 1.5 x IQR`), also per year-month. Print how many hours are flagged by: both methods, only the z-score method, and only the IQR method.

b) Find the "problematic forecast" hours where the absolute `load_forecast_error` is more than 5% of `total load actual`. Print the number of problematic hours per year.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.


Compare the two anomaly detection methods. Which one is more appropriate for `price actual`, and why? (Hint: look back at your answer about skewness in question 4.) Write your answer in the Markdown cell below.

#### START.
Your analysis / interpretation goes here:



#### END.

## 12: Correlation analysis

For the variables `price actual`, `price day ahead`, `total load actual`, `generation fossil gas`, `generation fossil hard coal`, `generation fossil brown coal/lignite`, `generation nuclear`, `generation wind onshore`, `generation solar`, `generation hydro water reservoir` and `renewable_share`:

- calculate the **Pearson** and the **Spearman** correlation matrices and show them as two annotated seaborn heatmaps, side by side, using a diverging colour map centred at 0
- calculate the correlation between `price actual` and `generation wind onshore` separately for **winter** (December-February) and **summer** (June-August)

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

#### START.
Describe the outputs. When do Pearson and Spearman differ, and why? Does a strong correlation show that one variable causes the other?



#### END.

# Section D: Big Data Analysis with PySpark

In this section you will use PySpark and the `fuel_in_france_2026_01_to_2026_09` dataset (either .csv or .parquet).

Note 1: Work through each question in order (later questions build on DataFrames and columns created in earlier ones).

Note 2: After question 19, there is a single Markdown cell for your analysis and interpretation of findings for questions 15-19.

### Start a Spark Session
Every PySpark script needs a `SparkSession`. This is the entry point to using Spark. Create a session named as your student ID. For instance, if your student ID is 12345678, the session must be named `12345678`.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

## 13: Loading the fuel dataset and defining a schema

a) Load the dataset into a Spark DataFrame named `df_raw`, using `header=True` and `inferSchema=True`. Print its schema and measure the time taken to load it.

b) Now define an explicit schema (`StructType`) where `station_id`, `departement`, `regioncode` and `zipcode` are `StringType`, `rate_date` and `brent_date` are `StringType` (they are converted for you after question 14), and all numerical columns are `DoubleType`. Load the dataset again using your schema into a Spark DataFrame named `df_fuel`. Print its schema and measure the time taken to load it.

c) Some `zipcode` values have lost their leading `0` (for example `1000` instead of `01000`). Fix them in `df_fuel` so that every zipcode has 5 characters (hint: `F.lpad`).

d) Print the total number of rows and the number of partitions of `df_fuel`.

Note: if you choose the Parquet file, `header` and `inferSchema` do not apply, because Parquet stores its own schema. In (a), load the file and print its stored schema; in (b), apply your explicit schema by casting the columns (`.cast()`).

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.



Compare the two schemas and the two loading times. Why is an explicit schema important for this dataset (hint: look at `zipcode` and `departement`)? Write your answer in the Markdown cell below.

#### START.
Your analysis / interpretation goes here:



#### END.

## 14: Understanding the data and data quality

Using `df_fuel`, report:
- the number of missing values in each column, calculated in a **single pass** over the data (hint: something like `F.sum(F.col(c).isNull().cast("int"))` for every column inside one `.select()`)
- the number of distinct stations and the number of distinct brands (`marque`)
- the number of rows for each `fuel_name`
- the number of exact duplicate rows

Then remove the exact duplicate rows, and remove rows where `price` is missing, less than or equal to 0, or greater than 5 (EUR per litre). Save the result back to `df_fuel` and print the new number of rows.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.



Considering this dataset, what other exploratory data analysis would you do to have a better understanding of the dataset? Write them as bullet points in the Markdown cell below.

#### START.
Your analysis / interpretation goes here:



#### END.

### Provided code: date conversion (not marked)

`rate_date` and `brent_date` are stored as text, and the date format is not the same across the whole dataset (for example `02/01/2026 00:01` in some rows and `2026-01-02 06:09:09` in others). Run the cell below **without changing it**. It adds the columns you need in the next questions:

- `rate_ts` (`TimestampType`): date and time of the price record
- `rate_day` (`DateType`): date of the price record
- `day_of_week` and `hour`: derived from `rate_ts`
- `brent_date` is converted into `DateType`

It also caches `df_fuel` in memory, as the next questions reuse it many times.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.



## 15: Aggregation - fuel types and geography

a) For each `fuel_name`, calculate: number of records, number of distinct stations, mean, standard deviation, minimum and maximum `price` (rounded to 3 decimal places). Sort by mean price.

b) For `Gazole` and `E10` only, calculate the mean `price` for each `regioncode`, and pivot the result so that each fuel is a column. Sort by the mean `Gazole` price, descending.

c) Considering only departments with at least 30 distinct stations, which `departement` has the highest mean `Gazole` price, and which has the lowest?

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

## 16: Brand comparison

For `Gazole` only, calculate the mean `price` and the number of distinct stations for each `marque`. Keep only brands with at least 50 distinct stations. Show the 5 cheapest and the 5 most expensive brands.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

## 17: Time-based analysis with window functions

a) Calculate the daily national mean price for each `fuel_name` and each `rate_day`. Save the result in a DataFrame named `daily_prices`.

b) For each `fuel_name`, calculate a 7-day (trailing, inclusive) moving average of the daily mean price, ordered by `rate_day`. Add it to `daily_prices` in a column named `ma_7_price`. Show the result for `Gazole` only.

c) For each station and fuel, find its **most recent** price record. Save the result in a DataFrame named `latest_prices` (one row per `station_id` and `fuel_name`). Print its number of rows and show 10 rows.

d) Convert `daily_prices` (a small, aggregated DataFrame) into pandas and plot the daily mean price and `ma_7_price` of each fuel over time.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.



## 18: Anomaly detection at scale

A price may be normal for one fuel type and unusual for another, and prices change day by day. For each `fuel_name` and `rate_day`, calculate the mean and standard deviation of `price`, join them back to `df_fuel`, and calculate a z-score for every record in a new column `price_z`. Define "unusual" as records where `|price_z| > 3`.

- print the number of unusual records for each `fuel_name`
- show the 10 most extreme unusual records, including `nom`, `commune`, `marque`, `fuel_name`, `price` and `price_z`
- save all unusual records in Parquet format, in a folder named `fuel_price_anomalies`

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.



## 19: Correlation - Brent crude oil and pump prices

a) Create a daily DataFrame with one row per `rate_day`, holding the national mean `Gazole` price, the national mean `E10` price and the `brent_rate_eur` for that day. Calculate the correlation between the mean `Gazole` price and `brent_rate_eur`, and between the mean `E10` price and `brent_rate_eur`.

b) Pump prices may react to crude oil prices with a delay. Using a window function (`F.lag`), calculate the correlation between the mean `Gazole` price on each day and `brent_rate_eur` from *k* days earlier, for *k* = 0, 1, 2, ..., 5. Which lag gives the strongest correlation?

Note: `brent_rate_eur` is missing on days without a Brent price (e.g. weekends). Make sure these days do not break your calculation.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

Your analysis / interpretation for PySpark tasks (15-19) go here. Discuss the results of the aggregations, the brand and regional differences, the anomalies you detected and the relationship between Brent and pump prices. Consider the limitations of the dataset and of your analysis.

#### START.
Your analysis / interpretation goes here:



#### END.

## 20: Storage

Write `df_fuel` to Parquet format, partitioned by `fuel_name`, in a folder named `fuel_parquet`.

In [ ]:
# BEGIN SOLUTION
# Your code here with comments or explanation


# END SOLUTION

In [ ]:
# Add at least one test case here to verify the correctness of your code.
# You can use assertions or print statements to check the expected output.



#### START.
Your analysis / interpretation goes here:



#### END.

## 21: Your Final Reflection

In less than 800 words, reflect on how you approached this assessment, what was new to you and how you found solutions. Explain how you acted on the formative feedback you received during the module, and what you changed as a result.

Moreover, relate your experience on this assessment to Big Data's four Vs (volume, variety, velocity, and veracity). Choose 2 of them and give one concrete example from your own implementation for each (at least one example must come from Section D).

Finally, choose one of the UN Sustainable Development Goals (e.g. SDG 7 - Affordable and Clean Energy, or SDG 13 - Climate Action) and briefly explain how the kind of analysis you did in this assessment could support it.

#### START
Your reflection here:




#### END